<a href="https://colab.research.google.com/github/alancheng-1/leadopt_agents/blob/main/adverserial_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import json
import re
from typing import List, Dict

# Mock LLM API call to represent agent interactions
def mock_llm_agent(system_prompt: str, user_prompt: str) -> str:
    """
    Simulates an LLM agent call.
    In production, this would call the Google Gemini or internal LLM API.
    """
    # Simple rule-based generation to simulate the adversarial loop
    if "Generative Medicinal Chemist" in system_prompt:
        if "REJECTED" in user_prompt:
            # The agent is reacting to feedback and trying to "repair" the molecule
            return json.dumps({
                "SMILES": "CC1=NC(C(F)(F)F)=CS1",  # Swaps toxic Cl for CF3 group
                "Hypothesis": "Replaced the chloromethyl group with a stable trifluoromethyl group to avoid thiol-reactivity while maintaining hydrophobic pocket binding.",
                "Synthetic_Steps": [
                    "1. Utilize a Ruppert-Prakash reagent for trifluoromethylation.",
                    "2. Followed by condensation with mercaptoacetone."
                ]
            }, indent=2)
        else:
            # First draft proposal
            return json.dumps({
                "SMILES": "CC1=NC(CCl)=CS1",  # Toxic, reactive intermediate
                "Hypothesis": "Thiazole core with a chloromethyl group designed to access a deep hydrophobic cleft in the binding pocket.",
                "Synthetic_Steps": [
                    "1. React chloroacetonitrile with sodium methoxide.",
                    "2. Condense with mercaptoacetone to yield the core thiazole structure."
                ]
            }, indent=2)

    elif "Adversarial ADMET/Tox Critic" in system_prompt:
        smiles = re.search(r'"SMILES":\s*"([^"]+)"', user_prompt)
        if smiles and "CCl" in smiles.group(1):
            return json.dumps({
                "Decision": "REJECTED",
                "Reason": "The chloromethyl group (CCl) is a highly reactive alkylating agent, presenting severe mutagenicity, toxicity, and chemical instability in vivo.",
                "Recommended_Action": "Substitute the alkyl halide with a bioisosteric fluorine or trifluoromethyl (-CF3) group to maintain pocket occupancy safely."
            }, indent=2)
        return json.dumps({"Decision": "APPROVED", "Reason": "SMILES profile has safe toxicity boundaries."})

    elif "Adversarial Synthetability Critic" in system_prompt:
        return json.dumps({"Decision": "APPROVED", "Reason": "Proposed synthetic steps leverage well-documented reagents."})

    return "{}"


class AdversarialChemistryHarness:
    def __init__(self):
        self.generator_prompt = (
            "You are a Generative Medicinal Chemist Agent. Your task is to output a new molecular design "
            "as a JSON object containing 'SMILES', 'Hypothesis', and 'Synthetic_Steps'. "
            "If you receive a rejection, you must repair your SMILES based on the feedback."
        )
        self.tox_critic_prompt = (
            "You are an Adversarial ADMET/Tox Critic. Your sole job is to reject molecules with toxic groups, "
            "metabolic liabilities, or electrophilic reactive centers. Output a JSON with 'Decision' (APPROVED/REJECTED) "
            "and 'Reason'/'Recommended_Action'."
        )
        self.synth_critic_prompt = (
            "You are an Adversarial Synthetability Critic. Review the synthetic route and reject if the reactions "
            "are physically impossible or rely on hazardous/unavailable reagents."
        )

    def run_optimization_cycle(self, target_objective: str, max_iterations: int = 3) -> Dict:
        print(f"Starting Adversarial Design Loop for: {target_objective}\n")

        current_input = f"Design a potent small molecule inhibitor targeting: {target_objective}"

        for iteration in range(1, max_iterations + 1):
            print(f"--- Iteration {iteration} ---")

            # 1. GENERATION STEP
            raw_proposal = mock_llm_agent(self.generator_prompt, current_input)
            proposal = json.loads(raw_proposal)
            print(f"[Generator] Proposed SMILES: {proposal['SMILES']}")
            print(f"[Generator] Hypothesis: {proposal['Hypothesis']}\n")

            # 2. ADVERSARIAL CRITIQUE STEP
            tox_feedback_raw = mock_llm_agent(self.tox_critic_prompt, raw_proposal)
            tox_feedback = json.loads(tox_feedback_raw)

            synth_feedback_raw = mock_llm_agent(self.synth_critic_prompt, raw_proposal)
            synth_feedback = json.loads(synth_feedback_raw)

            # 3. EVALUATE FEEDBACK
            is_valid = True
            critique_payload = {}

            if tox_feedback.get("Decision") == "REJECTED":
                print(f"[Tox Critic] REJECTED!")
                print(f"Reason: {tox_feedback['Reason']}")
                print(f"Recommendation: {tox_feedback['Recommended_Action']}\n")
                is_valid = False
                critique_payload["Tox_Feedback"] = tox_feedback

            if synth_feedback.get("Decision") == "REJECTED":
                print(f"[Synthesis Critic] REJECTED!")
                is_valid = False
                critique_payload["Synth_Feedback"] = synth_feedback

            if is_valid:
                print("[Adversarial Harness] ALL CRITICS PASSED!")
                print("Molecule approved for wet-lab synthesis hand off.\n")
                return proposal

            # Set up the "repair" input context for the generator in the next loop
            current_input = (
                f"REJECTED: Your previous design {proposal['SMILES']} failed the validation harness. "
                f"Feedback details: {json.dumps(critique_payload)}"
            )

        print("Optimization loop completed without full approval.")
        return proposal


# Run the simulation
if __name__ == "__main__":
    harness = AdversarialChemistryHarness()
    approved_molecule = harness.run_optimization_cycle("Kinase inhibitor with strong pocket occupancy")
    print("\nFinal Handoff Molecular Entry:")
    print(json.dumps(approved_molecule, indent=4))


Starting Adversarial Design Loop for: Kinase inhibitor with strong pocket occupancy

--- Iteration 1 ---
[Generator] Proposed SMILES: CC1=NC(CCl)=CS1
[Generator] Hypothesis: Thiazole core with a chloromethyl group designed to access a deep hydrophobic cleft in the binding pocket.

[Tox Critic] REJECTED!
Reason: The chloromethyl group (CCl) is a highly reactive alkylating agent, presenting severe mutagenicity, toxicity, and chemical instability in vivo.
Recommendation: Substitute the alkyl halide with a bioisosteric fluorine or trifluoromethyl (-CF3) group to maintain pocket occupancy safely.

--- Iteration 2 ---
[Generator] Proposed SMILES: CC1=NC(C(F)(F)F)=CS1
[Generator] Hypothesis: Replaced the chloromethyl group with a stable trifluoromethyl group to avoid thiol-reactivity while maintaining hydrophobic pocket binding.

[Adversarial Harness] ALL CRITICS PASSED!
Molecule approved for wet-lab synthesis hand off.


Final Handoff Molecular Entry:
{
    "SMILES": "CC1=NC(C(F)(F)F)=CS1",


In [12]:
# Run the simulation
if __name__ == "__main__":
    harness = AdversarialChemistryHarness()
    # UPDATED LINE:
    approved_molecule = harness.run_optimization_cycle("Protease inhibitor with strong pocket occupancy", max_iterations=10)


Starting Adversarial Design Loop for: Protease inhibitor with strong pocket occupancy

--- Iteration 1 ---
[Generator] Proposed SMILES: CC1=NC(CCl)=CS1
[Generator] Hypothesis: Thiazole core with a chloromethyl group designed to access a deep hydrophobic cleft in the binding pocket.

[Tox Critic] REJECTED!
 Reason: The chloromethyl group (CCl) is a highly reactive alkylating agent, presenting severe mutagenicity, toxicity, and chemical instability in vivo.
 Recommendation: Substitute the alkyl halide with a bioisosteric fluorine or trifluoromethyl (-CF3) group to maintain pocket occupancy safely.

--- Iteration 2 ---
[Generator] Proposed SMILES: CC1=NC(C(F)(F)F)=CS1
[Generator] Hypothesis: Replaced the chloromethyl group with a stable trifluoromethyl group to avoid thiol-reactivity while maintaining hydrophobic pocket binding.

[Adversarial Harness] ALL CRITICS PASSED!
Molecule approved for wet-lab synthesis hand off.

